# YOLO Apple Defect Detection Training

**Kaggle / CPU mode** - GPU auto detect

Steps:
1. Upload yolo_detect_dataset.zip as a Kaggle Dataset
2. Accelerator = None (CPU) or GPU if available
3. CPU: epochs=20, imgsz=320, batch=4
4. GPU: epochs=50, imgsz=640, batch=16
5. Download best.pt to E:\apple\yolo_detect_best.pt


In [ ]:
!pip install ultralytics -q

In [ ]:
import os, shutil, time, glob, zipfile

# Auto-find the dataset in /kaggle/input/
input_dir = "/kaggle/input"
dataset_root = None
for root, dirs, files in os.walk(input_dir):
    if "data.yaml" in files:
        dataset_root = root
        break

if dataset_root is None:
    # Try to find a zip file and extract
    zips = glob.glob(os.path.join(input_dir, "**", "*.zip"), recursive=True)
    if zips:
        print(f"Found zip: {zips[0]}, extracting...")
        with zipfile.ZipFile(zips[0], 'r') as zf:
            zf.extractall("/content/yolo_detect_dataset")
        dataset_root = "/content/yolo_detect_dataset"
    else:
        print("Dataset not found in /kaggle/input/")
        print("Please upload yolo_detect_dataset.zip as a Kaggle Dataset first.")
        raise FileNotFoundError("Dataset not found")

print(f"Dataset root: {dataset_root}")
os.system(f"ls -la {dataset_root}")

In [ ]:
import torch
from ultralytics import YOLO
import os, time, glob

input_dir = '/kaggle/input'
dataset_root = None
for root, dirs, files in os.walk(input_dir):
    if 'data.yaml' in files:
        dataset_root = root
        break
if dataset_root is None:
    raise FileNotFoundError('Upload dataset to Kaggle first')
os.system('ls -la ' + dataset_root)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
is_cpu = device == 'cpu'
print(f'Device: {device}')

model = YOLO('yolo11n.pt')
t0 = time.time()
model.train(
    data=os.path.join(dataset_root, 'data.yaml'),
    epochs=20 if is_cpu else 50,
    imgsz=320 if is_cpu else 640,
    batch=4 if is_cpu else 16,
    device=device,
    project='/kaggle/working/yolo_detect_output',
    name='apple_detect', exist_ok=True,
    patience=10 if is_cpu else 20,
    augment=not is_cpu,
    workers=2, cos_lr=True, warmup_epochs=3,
)
sec = time.time() - t0
print(f'Done! {sec/60:.1f} min')


In [ ]:
best = '/kaggle/working/yolo_detect_output/apple_detect/weights/best.pt'
if os.path.exists(best):
    import shutil
    shutil.copy2(best, '/kaggle/working/best.pt')
    size_mb = os.path.getsize(best) / 1e6
    print(f'Model: /kaggle/working/best.pt ({size_mb:.1f} MB)')
else:
    print('Model not found!')
